<a href="https://colab.research.google.com/github/edwardoughton/IGARSS26/blob/main/notebook_3_testing_validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🛰️ IGARSS 2026 Summer School — Part 3 of 3 🛰️
## AI Refusnik to AI Evangelist: Excellent at AI-Assisted Coding for Satellite Image Analysis

**Instructor:** Prof. Edward Oughton, George Mason University

**Session duration:** ~60 minutes

---

Welcome to the final part of this summer school!

In Part 3 we close the loop: having used AI agents to build satellite image processing pipelines (Parts 1–2), we now focus on **testing, validation, and evaluation** — the practices that make AI-assisted research *credible*.

By the end of Part 3, you will be able to:
- Write **unit tests** for geospatial processing functions using `pytest`
- Construct **synthetic test data** that mimics satellite imagery characteristics
- Implement **accuracy assessment** for a land cover classification using a confusion matrix
- Apply **cross-validation** strategies for spatially autocorrelated data
- Quantify and communicate **uncertainty** in spectral index products
- Understand **reproducibility** best practices for AI-assisted geoscience research

## 📗 Learning Objectives 📗

By the end of this notebook, you should be able to:

1. Explain why testing is *especially* important when using AI-generated code
2. Write `pytest`-style unit tests for remote sensing processing functions
3. Build synthetic satellite image test fixtures with known properties
4. Compute and interpret an **accuracy assessment** (overall, producer's, user's accuracy; kappa)
5. Apply a **spatial cross-validation** approach to avoid information leakage
6. Estimate and visualize **propagated uncertainty** in spectral indices
7. Document an AI-assisted pipeline in a reproducible, citation-ready format

---

## 0. Why Testing Matters More When You Use AI

When a human expert writes code, they often build an implicit understanding of the logic that allows them to spot errors by inspection. When an AI writes code:

- The code may look correct but contain subtle scientific errors
- The AI may use outdated API signatures
- The AI may hallucinate function arguments or default values
- The AI's training data may not include the latest sensor calibration updates

**Testing transforms these risks from latent bugs into visible failures.**

### The three-layer testing pyramid for geospatial code:

```
         ┌────────────────────────────────┐
         │      Integration tests         │  ← Full pipeline with real data
         │   (slow, hard to isolate)      │
         ├────────────────────────────────┤
         │       Function tests           │  ← Test individual functions
         │  (medium speed, clear errors)  │
         ├────────────────────────────────┤
         │         Unit tests             │  ← Test mathematical properties
         │    (fast, highly isolated)     │
         └────────────────────────────────┘
```

In this session we focus primarily on **unit tests** and **function tests**, with a brief integration test at the end.

---

## 1. Install and import dependencies

In [ ]:
!pip -q install pytest pytest-cov numpy matplotlib scikit-learn scipy rasterio geopandas

In [ ]:
import os
import json
import warnings
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import rasterio
from rasterio.transform import from_origin
from rasterio.crs import CRS

from sklearn.metrics import (
    confusion_matrix, classification_report,
    cohen_kappa_score, accuracy_score
)
from scipy import ndimage

warnings.filterwarnings('ignore')

DATA_DIR = Path('igarss26_data')
DATA_DIR.mkdir(exist_ok=True)

print('Packages imported.')

---

## 2. Re-implement the core functions from Parts 1 and 2

We copy the functions we built across Parts 1 and 2 into a single module so we can test them systematically. In a real research project, these would live in a Python package (e.g., `src/eo_pipeline.py`).

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Core functions (as produced and validated in Parts 1 & 2)
# ═══════════════════════════════════════════════════════════════

def safe_index(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    """
    Compute the normalized difference index (a - b) / (a + b).

    Returns NaN where the denominator is zero to prevent division errors.

    Parameters
    ----------
    a, b : np.ndarray
        Input 2D float arrays (same shape).

    Returns
    -------
    np.ndarray
        Normalized difference array, same shape as inputs.
    """
    denom = a + b
    return np.where(denom == 0, np.nan, (a - b) / denom)


def clip_reflectance(band: np.ndarray) -> np.ndarray:
    """Clip reflectance values to the physically valid range [0, 1]."""
    return np.clip(band, 0, 1)


def detect_ndvi_change(
    ndvi_t1: np.ndarray,
    ndvi_t2: np.ndarray,
    threshold: float = 0.15
):
    """
    Detect significant NDVI change between two time periods.

    Returns delta_ndvi (smoothed difference) and change_class
    (+1=greening, 0=stable, -1=browning, -9=nodata).
    """
    valid = np.isfinite(ndvi_t1) & np.isfinite(ndvi_t2)
    delta = np.full_like(ndvi_t1, np.nan)
    delta[valid] = ndvi_t2[valid] - ndvi_t1[valid]

    delta_filled = np.where(valid, delta, 0)
    delta_smooth = ndimage.median_filter(delta_filled, size=3)
    delta_smooth = np.where(valid, delta_smooth, np.nan)

    change_class = np.full_like(delta_smooth, -9, dtype=np.int8)
    change_class[valid & (delta_smooth >  threshold)] =  1
    change_class[valid & (delta_smooth < -threshold)] = -1
    change_class[valid & (np.abs(delta_smooth) <= threshold)] = 0

    return delta_smooth, change_class


print('Core functions defined.')

---

## 3. Building Synthetic Test Fixtures

A **test fixture** is a controlled dataset with **known properties** that allows us to verify that our functions behave correctly. For satellite image functions, we construct synthetic arrays that simulate specific land cover types.

### Why synthetic data for testing?

1. We control the ground truth — we *know* what the output should be
2. Tests run fast (no network calls, no large file I/O)
3. We can test **edge cases** (all zeros, all NaN, extreme values) that rarely appear in real data
4. Tests are reproducible and platform-independent

In [ ]:
def make_landcover_reflectance(n_rows: int = 100, n_cols: int = 100, seed: int = 42):
    """
    Generate a synthetic Landsat-like reflectance scene with four distinct
    land cover regions, each with known spectral characteristics.

    Regions (by quadrant):
      Top-left     : Water (low red, low NIR, negative NDVI)
      Top-right    : Dense vegetation (low red, high NIR, high NDVI)
      Bottom-left  : Urban/built-up (moderate red, low NIR, low NDVI)
      Bottom-right : Bare soil (high red, moderate NIR, low NDVI)

    Returns
    -------
    dict
        Keys 'red', 'green', 'blue', 'nir', 'swir1' map to 2D float arrays
        of surface reflectance in [0, 1].
    dict
        'true_labels': 2D int array (0=water, 1=veg, 2=urban, 3=bare)
        for accuracy assessment.
    """
    rng = np.random.default_rng(seed)
    half_r, half_c = n_rows // 2, n_cols // 2

    def noisy(base, noise, shape):
        return np.clip(rng.normal(base, noise, shape), 0, 1)

    # Reflectance signatures (R, G, B, NIR, SWIR1)
    signatures = {
        'water': {'red': 0.04, 'green': 0.06, 'blue': 0.10, 'nir': 0.02, 'swir1': 0.01},
        'veg':   {'red': 0.05, 'green': 0.10, 'blue': 0.04, 'nir': 0.45, 'swir1': 0.12},
        'urban': {'red': 0.18, 'green': 0.17, 'blue': 0.16, 'nir': 0.14, 'swir1': 0.20},
        'bare':  {'red': 0.30, 'green': 0.25, 'blue': 0.20, 'nir': 0.22, 'swir1': 0.28},
    }
    noise = 0.02  # Band noise sigma

    bands = {b: np.zeros((n_rows, n_cols)) for b in ['red', 'green', 'blue', 'nir', 'swir1']}
    labels = np.zeros((n_rows, n_cols), dtype=np.int8)

    quadrant_map = [
        (slice(None, half_r),  slice(None, half_c),  'water', 0),
        (slice(None, half_r),  slice(half_c, None),  'veg',   1),
        (slice(half_r, None),  slice(None, half_c),  'urban', 2),
        (slice(half_r, None),  slice(half_c, None),  'bare',  3),
    ]

    for (rs, cs, lc_name, label) in quadrant_map:
        shape = (rs.stop or n_rows) - (rs.start or 0), (cs.stop or n_cols) - (cs.start or 0)
        for band in bands:
            bands[band][rs, cs] = noisy(signatures[lc_name][band], noise, shape)
        labels[rs, cs] = label

    # Introduce a small NaN patch to simulate cloud masking
    bands['red'][40:45, 40:45] = np.nan
    bands['nir'][40:45, 40:45] = np.nan

    return bands, {'true_labels': labels}


bands, ground_truth = make_landcover_reflectance(n_rows=100, n_cols=100)

print('Synthetic test scene created.')
print(f'Shape: {bands["red"].shape}')
print(f'True classes: {np.unique(ground_truth["true_labels"])}')
print(f'NaN pixels in red band: {np.isnan(bands["red"]).sum()}')

In [ ]:
# Visualize the synthetic scene
def stretch(arr, low=2, high=98):
    lo, hi = np.nanpercentile(arr, [low, high])
    return np.clip((arr - lo) / (hi - lo + 1e-9), 0, 1)

rgb = np.dstack([
    stretch(bands['red']),
    stretch(bands['green']),
    stretch(bands['blue'])
])
rgb = np.nan_to_num(rgb, nan=0.0)

label_colors = np.zeros((100, 100, 3))
label_colors[ground_truth['true_labels'] == 0] = [0.1, 0.2, 0.9]  # water
label_colors[ground_truth['true_labels'] == 1] = [0.0, 0.6, 0.0]  # veg
label_colors[ground_truth['true_labels'] == 2] = [0.7, 0.7, 0.7]  # urban
label_colors[ground_truth['true_labels'] == 3] = [0.8, 0.6, 0.3]  # bare

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(rgb)
axes[0].set_title('Synthetic True-Colour', fontsize=11)
axes[0].axis('off')

axes[1].imshow(label_colors)
axes[1].set_title('Ground Truth Labels', fontsize=11)
axes[1].axis('off')
legend_patches = [
    mpatches.Patch(color=[0.1, 0.2, 0.9], label='Water'),
    mpatches.Patch(color=[0.0, 0.6, 0.0], label='Dense Vegetation'),
    mpatches.Patch(color=[0.7, 0.7, 0.7], label='Urban'),
    mpatches.Patch(color=[0.8, 0.6, 0.3], label='Bare Soil'),
]
axes[1].legend(handles=legend_patches, loc='lower right', fontsize=8)

plt.suptitle('Synthetic Landsat Test Scene', fontsize=12)
plt.tight_layout()
plt.show()

---

## 4. Writing Unit Tests with pytest

We write **pytest**-style tests directly in the notebook. In a real project these would live in `tests/test_eo_pipeline.py`.

Key pytest conventions:
- Test functions start with `test_`
- Use `assert` for comparisons
- Use `np.testing.assert_allclose` for floating-point comparisons with tolerance
- We use `ipytest` to run pytest inside Jupyter

In [ ]:
!pip -q install ipytest
import ipytest
ipytest.autoconfig()
print('ipytest configured.')

In [ ]:
%%ipytest -v

# ─── Unit tests for safe_index ───────────────────────────────────

def test_safe_index_known_values():
    """safe_index should produce exact NDVI for simple cases."""
    # For NIR=0.5, Red=0.1: NDVI = (0.5-0.1)/(0.5+0.1) = 0.4/0.6 ≈ 0.6667
    nir = np.array([[0.5]])
    red = np.array([[0.1]])
    result = safe_index(nir, red)
    np.testing.assert_allclose(result, [[0.4/0.6]], rtol=1e-5)


def test_safe_index_zero_denominator():
    """safe_index must return NaN (not raise) when denominator is zero."""
    a = np.array([[0.0, 0.5]])
    b = np.array([[0.0, 0.5]])
    result = safe_index(a, b)
    assert np.isnan(result[0, 0]), 'Expected NaN for zero denominator'
    np.testing.assert_allclose(result[0, 1], 0.0)


def test_safe_index_range():
    """NDVI output must lie within [-1, 1] for non-NaN valid reflectance."""
    rng = np.random.default_rng(0)
    a = rng.uniform(0, 1, (50, 50)).astype(np.float32)
    b = rng.uniform(0, 1, (50, 50)).astype(np.float32)
    result = safe_index(a, b)
    valid = result[np.isfinite(result)]
    assert valid.min() >= -1.0 - 1e-6
    assert valid.max() <=  1.0 + 1e-6


def test_safe_index_symmetry():
    """safe_index(a, b) == -safe_index(b, a) for non-zero denominator."""
    a = np.array([[0.3, 0.6]])
    b = np.array([[0.6, 0.2]])
    forward = safe_index(a, b)
    backward = safe_index(b, a)
    np.testing.assert_allclose(forward, -backward, rtol=1e-6)


def test_safe_index_preserves_shape():
    """Output shape must match input shape."""
    a = np.ones((30, 40))
    b = np.ones((30, 40)) * 0.5
    result = safe_index(a, b)
    assert result.shape == (30, 40)


# ─── Unit tests for clip_reflectance ─────────────────────────────

def test_clip_reflectance_positive_values():
    """Values above 1.0 should be clipped to 1.0."""
    arr = np.array([[-0.1, 0.5, 1.5, 2.0]])
    result = clip_reflectance(arr)
    expected = np.array([[0.0, 0.5, 1.0, 1.0]])
    np.testing.assert_array_equal(result, expected)


def test_clip_reflectance_valid_passthrough():
    """Values already in [0, 1] should pass through unchanged."""
    arr = np.array([[0.0, 0.25, 0.5, 0.75, 1.0]])
    result = clip_reflectance(arr)
    np.testing.assert_array_equal(result, arr)

In [ ]:
%%ipytest -v

# ─── Unit tests for detect_ndvi_change ───────────────────────────

def test_detect_ndvi_change_greening():
    """Large positive ΔNDVI should produce class +1 (greening)."""
    # t1 low, t2 high → strong greening
    ndvi_t1 = np.full((10, 10), 0.1, dtype=float)
    ndvi_t2 = np.full((10, 10), 0.7, dtype=float)
    _, change = detect_ndvi_change(ndvi_t1, ndvi_t2, threshold=0.15)
    # Most pixels (excluding median filter edge effects) should be +1
    assert (change[1:-1, 1:-1] == 1).all(), 'Interior pixels should all be greening'


def test_detect_ndvi_change_browning():
    """Large negative ΔNDVI should produce class -1 (browning)."""
    ndvi_t1 = np.full((10, 10), 0.7, dtype=float)
    ndvi_t2 = np.full((10, 10), 0.1, dtype=float)
    _, change = detect_ndvi_change(ndvi_t1, ndvi_t2, threshold=0.15)
    assert (change[1:-1, 1:-1] == -1).all(), 'Interior pixels should all be browning'


def test_detect_ndvi_change_stable():
    """Small ΔNDVI should produce class 0 (stable)."""
    ndvi_t1 = np.full((10, 10), 0.4, dtype=float)
    ndvi_t2 = np.full((10, 10), 0.41, dtype=float)  # delta = 0.01 < threshold
    _, change = detect_ndvi_change(ndvi_t1, ndvi_t2, threshold=0.15)
    assert (change[1:-1, 1:-1] == 0).all(), 'Interior pixels should be stable'


def test_detect_ndvi_change_nodata_propagation():
    """NaN in either input array should result in nodata (-9) in output."""
    ndvi_t1 = np.full((5, 5), 0.4, dtype=float)
    ndvi_t2 = np.full((5, 5), 0.6, dtype=float)
    # Introduce NaN in a single pixel of t1
    ndvi_t1[2, 2] = np.nan
    _, change = detect_ndvi_change(ndvi_t1, ndvi_t2, threshold=0.15)
    assert change[2, 2] == -9, 'NaN pixel should map to nodata (-9) in output'


def test_detect_ndvi_change_output_shape():
    """Output shape must match input shape."""
    ndvi_t1 = np.random.rand(20, 30)
    ndvi_t2 = np.random.rand(20, 30)
    delta, change = detect_ndvi_change(ndvi_t1, ndvi_t2)
    assert delta.shape == (20, 30)
    assert change.shape == (20, 30)


def test_detect_ndvi_change_threshold_symmetry():
    """Pixels at exactly the threshold should be classified as stable (class 0)."""
    ndvi_t1 = np.full((5, 5), 0.3, dtype=float)
    ndvi_t2 = np.full((5, 5), 0.45, dtype=float)  # delta = 0.15 exactly
    _, change = detect_ndvi_change(ndvi_t1, ndvi_t2, threshold=0.15)
    # delta == threshold → should be stable (0), not greening (1)
    assert (change[1:-1, 1:-1] == 0).all(), 'Pixels at threshold should be classified as stable'

### 🏆 Expected outcome

All tests above should pass (`PASSED` in green). If any fail, it means our implementation contains a bug — and the test message tells us precisely *which* condition is violated. This is the power of unit testing: bugs surface immediately, not months later when results look suspicious.

### ✏️ Exercise 4.1 — Write your own tests

Add at least two more tests for `safe_index`:

1. Test the behaviour with all-NaN input arrays
2. Test that the output for a known water pixel (NIR ≈ 0.02, Red ≈ 0.04) gives negative NDVI

*Use the cell below.*

In [ ]:
%%ipytest -v

# Your tests here
def test_placeholder():
    """Replace this with your tests."""
    assert True

---

## 5. Accuracy Assessment for Land Cover Classification

Accuracy assessment is the standard method for evaluating the quality of a land cover map. We use the synthetic scene with **known ground truth** to demonstrate the full workflow.

### Key accuracy metrics:

| Metric | Formula | Interpretation |
|---|---|---|
| **Overall Accuracy (OA)** | Correctly classified / Total | Global correctness |
| **Producer's Accuracy (PA)** | True Positive / Actual Positive | Sensitivity per class |
| **User's Accuracy (UA)** | True Positive / Predicted Positive | Precision per class |
| **Cohen's Kappa (κ)** | Adjusted for chance agreement | 0=random, 1=perfect |

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

def kmeans_classify_synthetic(bands, n_clusters=4, random_state=42):
    """
    Apply K-Means classification to the synthetic reflectance scene
    using NDVI, NDWI, and NDBI as features.

    Returns
    -------
    classified : np.ndarray
        2D int array of class labels (0 to n_clusters-1), -1 for nodata.
    """
    red   = bands['red']
    green = bands['green']
    nir   = bands['nir']
    swir1 = bands['swir1']

    ndvi = safe_index(nir, red)
    ndwi = safe_index(green, nir)
    ndbi = safe_index(swir1, nir)

    rows, cols = ndvi.shape
    valid = np.isfinite(ndvi) & np.isfinite(ndwi) & np.isfinite(ndbi)

    features = np.column_stack([
        ndvi[valid], ndwi[valid], ndbi[valid]
    ])

    scaler = StandardScaler()
    features_scaled = scaler.fit_transform(features)

    km = KMeans(n_clusters=n_clusters, random_state=random_state, n_init=10)
    raw_labels = km.fit_predict(features_scaled)

    # Sort by mean NDVI ascending
    cluster_ndvi = [features[raw_labels == k, 0].mean() for k in range(n_clusters)]
    sorted_clusters = np.argsort(cluster_ndvi)
    remap = {old: new for new, old in enumerate(sorted_clusters)}
    sorted_labels = np.vectorize(remap.get)(raw_labels)

    classified = np.full((rows, cols), -1, dtype=np.int16)
    classified[valid] = sorted_labels
    return classified


# Run classification
predicted = kmeans_classify_synthetic(bands, n_clusters=4)

# Compare predicted vs ground truth
# We need to match class indices (K-Means labels are arbitrary orderings)
# Since we sort by NDVI: 0=water(low NDVI) → 3=dense veg(high NDVI)
# Ground truth: 0=water, 1=veg(high NDVI), 2=urban, 3=bare soil
# After NDVI sort mapping: water=0, urban≈1, bare≈2, veg=3

# Mask out nodata pixels for accuracy assessment
valid_pixels = predicted >= 0
y_pred = predicted[valid_pixels].ravel()
y_true = ground_truth['true_labels'][valid_pixels].ravel()

print(f'Total valid pixels for assessment: {len(y_pred):,}')
print(f'Predicted classes: {np.unique(y_pred)}')
print(f'True classes: {np.unique(y_true)}')

In [ ]:
def compute_accuracy_assessment(y_true, y_pred, class_names=None):
    """
    Compute a full accuracy assessment for a land cover classification.

    Calculates overall accuracy, producer's accuracy (recall),
    user's accuracy (precision), F1-score, and Cohen's Kappa.

    Parameters
    ----------
    y_true : array-like
        Ground truth class labels.
    y_pred : array-like
        Predicted class labels.
    class_names : list of str, optional
        Display names for each class.

    Returns
    -------
    pd.DataFrame
        Per-class accuracy metrics.
    float
        Overall accuracy.
    float
        Cohen's Kappa.
    """
    oa = accuracy_score(y_true, y_pred)
    kappa = cohen_kappa_score(y_true, y_pred)

    report = classification_report(
        y_true, y_pred,
        target_names=class_names,
        output_dict=True,
        zero_division=0
    )

    rows = []
    classes = class_names if class_names else sorted(set(y_true))
    for cls in classes:
        cls_key = str(cls)
        if cls_key in report:
            rows.append({
                'Class': cls,
                "User's Accuracy (Precision)": report[cls_key]['precision'],
                "Producer's Accuracy (Recall)": report[cls_key]['recall'],
                'F1-Score': report[cls_key]['f1-score'],
                'Support (n pixels)': int(report[cls_key]['support'])
            })

    df = pd.DataFrame(rows).set_index('Class').round(3)
    return df, oa, kappa


# K-Means assigns classes by NDVI order:
# 0=water(lowest NDVI), 1=urban, 2=bare, 3=veg(highest NDVI)
# Ground truth: 0=water, 1=veg, 2=urban, 3=bare
# Remap ground truth to match NDVI-sorted prediction order
gt_remap = {0: 0, 1: 3, 2: 1, 3: 2}   # water→0, veg→3, urban→1, bare→2
y_true_remapped = np.vectorize(gt_remap.get)(y_true)

class_names = ['Water', 'Urban', 'Bare Soil', 'Dense Veg']
df_acc, oa, kappa = compute_accuracy_assessment(y_true_remapped, y_pred, class_names)

print(f'\n=== Accuracy Assessment ===')
print(f'Overall Accuracy: {oa:.3f} ({oa*100:.1f}%)')
print(f"Cohen's Kappa:    {kappa:.3f}")
if kappa > 0.8:
    print('Kappa interpretation: Excellent agreement')
elif kappa > 0.6:
    print('Kappa interpretation: Good agreement')
elif kappa > 0.4:
    print('Kappa interpretation: Moderate agreement')
else:
    print('Kappa interpretation: Poor agreement')

print(f'\nPer-class metrics:')
print(df_acc.to_string())

In [ ]:
# Visualize the confusion matrix
cm = confusion_matrix(y_true_remapped, y_pred)
cm_normalized = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

im0 = axes[0].imshow(cm, cmap='Blues')
axes[0].set_xticks(range(len(class_names)))
axes[0].set_yticks(range(len(class_names)))
axes[0].set_xticklabels(class_names, rotation=45, ha='right', fontsize=9)
axes[0].set_yticklabels(class_names, fontsize=9)
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('True (Ground Truth)')
axes[0].set_title('Confusion Matrix (pixel counts)')
for i in range(len(class_names)):
    for j in range(len(class_names)):
        axes[0].text(j, i, str(cm[i, j]), ha='center', va='center', fontsize=10)
plt.colorbar(im0, ax=axes[0], fraction=0.046)

im1 = axes[1].imshow(cm_normalized, cmap='Blues', vmin=0, vmax=1)
axes[1].set_xticks(range(len(class_names)))
axes[1].set_yticks(range(len(class_names)))
axes[1].set_xticklabels(class_names, rotation=45, ha='right', fontsize=9)
axes[1].set_yticklabels(class_names, fontsize=9)
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('True (Ground Truth)')
axes[1].set_title('Normalized Confusion Matrix')
for i in range(len(class_names)):
    for j in range(len(class_names)):
        axes[1].text(j, i, f'{cm_normalized[i, j]:.2f}',
                     ha='center', va='center', fontsize=10)
plt.colorbar(im1, ax=axes[1], fraction=0.046)

plt.suptitle(f'Classification Accuracy Assessment  |  OA={oa:.3f}  κ={kappa:.3f}', fontsize=12)
plt.tight_layout()
plt.show()

---

## 6. Uncertainty Quantification in Spectral Indices

No satellite measurement is perfect. Sensor noise, atmospheric correction errors, and geometric registration introduce uncertainty into every pixel. Quantifying and communicating this uncertainty is essential for rigorous remote sensing science.

### Monte Carlo uncertainty propagation

We model band measurement uncertainty as Gaussian noise and propagate it through the NDVI formula:

$$\sigma_{\text{NDVI}}^2 \approx \left(\frac{\partial \text{NDVI}}{\partial \rho_{\text{NIR}}}\right)^2 \sigma_{\text{NIR}}^2 + \left(\frac{\partial \text{NDVI}}{\partial \rho_{\text{Red}}}\right)^2 \sigma_{\text{Red}}^2$$

For large ensembles, Monte Carlo simulation directly estimates the output distribution.

In [ ]:
def monte_carlo_ndvi_uncertainty(
    nir: np.ndarray,
    red: np.ndarray,
    sigma_nir: float = 0.005,
    sigma_red: float = 0.005,
    n_simulations: int = 500,
    seed: int = 42
):
    """
    Estimate NDVI uncertainty via Monte Carlo simulation.

    For each simulation, Gaussian noise is added independently to the
    NIR and Red bands before computing NDVI. The standard deviation
    of the resulting NDVI distribution is the uncertainty estimate.

    Parameters
    ----------
    nir, red : np.ndarray
        2D surface reflectance arrays.
    sigma_nir, sigma_red : float
        1-sigma reflectance uncertainty (unitless, same scale as inputs).
        Typical Landsat Level-2 absolute uncertainty ~ 0.005–0.01.
    n_simulations : int
        Number of Monte Carlo realizations.
    seed : int
        Random seed for reproducibility.

    Returns
    -------
    ndvi_mean : np.ndarray
        Mean NDVI across simulations.
    ndvi_std : np.ndarray
        Standard deviation (1-sigma uncertainty) of NDVI.
    """
    rng = np.random.default_rng(seed)
    ndvi_stack = np.zeros((n_simulations, *nir.shape))

    for i in range(n_simulations):
        nir_perturbed = np.clip(nir + rng.normal(0, sigma_nir, nir.shape), 0, 1)
        red_perturbed = np.clip(red + rng.normal(0, sigma_red, red.shape), 0, 1)
        ndvi_stack[i] = safe_index(nir_perturbed, red_perturbed)

    return np.nanmean(ndvi_stack, axis=0), np.nanstd(ndvi_stack, axis=0)


print('Running Monte Carlo uncertainty analysis...')

nir_syn  = bands['nir']
red_syn  = bands['red']

ndvi_mean, ndvi_uncertainty = monte_carlo_ndvi_uncertainty(
    nir_syn, red_syn, sigma_nir=0.008, sigma_red=0.008, n_simulations=300
)

print(f'Mean NDVI uncertainty (1σ): {np.nanmean(ndvi_uncertainty):.4f}')
print(f'Max NDVI uncertainty: {np.nanmax(ndvi_uncertainty):.4f}')

In [ ]:
# Visualize uncertainty
ndvi_deterministic = safe_index(nir_syn, red_syn)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

im0 = axes[0].imshow(ndvi_deterministic, cmap='RdYlGn', vmin=-0.2, vmax=0.8)
axes[0].set_title('NDVI (deterministic)', fontsize=11)
axes[0].axis('off')
plt.colorbar(im0, ax=axes[0], fraction=0.046)

im1 = axes[1].imshow(ndvi_mean, cmap='RdYlGn', vmin=-0.2, vmax=0.8)
axes[1].set_title('NDVI (MC mean, n=300)', fontsize=11)
axes[1].axis('off')
plt.colorbar(im1, ax=axes[1], fraction=0.046)

im2 = axes[2].imshow(ndvi_uncertainty, cmap='hot_r', vmin=0, vmax=0.05)
axes[2].set_title('NDVI Uncertainty (1σ)\nHigher near water/soil boundaries', fontsize=11)
axes[2].axis('off')
plt.colorbar(im2, ax=axes[2], fraction=0.046)

plt.suptitle('NDVI Monte Carlo Uncertainty Propagation\n'
             f'(σ_NIR = σ_Red = 0.008, n = 300 simulations)', fontsize=12)
plt.tight_layout()
plt.show()

# Where is uncertainty highest?
high_unc_mask = ndvi_uncertainty > np.nanpercentile(ndvi_uncertainty, 90)
print('\nHigh-uncertainty regions are typically:')
print('  - Near zero denominator (NIR + Red ≈ 0, e.g., water margins)')
print('  - Transition zones between land cover types')
print(f'  ({high_unc_mask.sum()} pixels in top 10% uncertainty)')

---

## 7. Reproducibility: Documenting Your AI-Assisted Pipeline

The final step in responsible AI-assisted research is **documentation and provenance tracking**. This section shows a minimal but complete template for recording how AI was used in your pipeline.

In [ ]:
import platform
import sys
import datetime

def generate_pipeline_report(output_path=None):
    """
    Generate a reproducibility report capturing environment, methods,
    AI tool usage, and validation outcomes for this pipeline run.

    This is the metadata record you should archive alongside your data products.
    """
    report = {
        'pipeline': 'IGARSS26 Satellite Image Analysis — AI-Assisted Pipeline',
        'version': '1.0',
        'timestamp': datetime.datetime.utcnow().isoformat() + 'Z',
        'environment': {
            'python': sys.version,
            'platform': platform.platform(),
            'key_packages': {
                'numpy': np.__version__,
                'rasterio': rasterio.__version__,
            }
        },
        'data_sources': {
            'catalog': 'Microsoft Planetary Computer STAC API',
            'catalog_url': 'https://planetarycomputer.microsoft.com/api/stac/v1',
            'landsat': {
                'collection': 'landsat-c2-l2',
                'processing_level': 'Level 2 Surface Reflectance',
                'scaling_factor': 10000,
                'citation': 'USGS (2023). Landsat Collection 2 Level-2 Science Products. '
                            'https://doi.org/10.5066/P9OGBGM6'
            },
            'sentinel2': {
                'collection': 'sentinel-2-l2a',
                'processing_level': 'Level 2A Bottom-of-Atmosphere Reflectance',
                'scaling_factor': 10000,
                'citation': 'ESA (2022). Sentinel-2 MSI Level-2A. '
                            'https://doi.org/10.5270/S2_-znk9xsj'
            }
        },
        'methods': {
            'spectral_indices': [
                {'name': 'NDVI', 'formula': '(NIR - Red) / (NIR + Red)'},
                {'name': 'NDWI', 'formula': '(Green - NIR) / (Green + NIR)'},
                {'name': 'NDBI', 'formula': '(SWIR1 - NIR) / (SWIR1 + NIR)'}
            ],
            'classification': {
                'algorithm': 'K-Means unsupervised clustering',
                'n_clusters': 5,
                'features': ['NDVI', 'NDWI', 'NDBI'],
                'preprocessing': 'StandardScaler (zero-mean, unit-variance)',
                'implementation': 'sklearn.cluster.KMeans'
            },
            'change_detection': {
                'method': 'NDVI differencing with spatial median filter',
                'threshold': 0.15,
                'spatial_filter': '3x3 median filter (scipy.ndimage.median_filter)'
            }
        },
        'ai_assistance': {
            'tools_used': ['OpenAI GPT-4o-mini (or compatible model)'],
            'usage_description': (
                'AI coding agents were used to generate initial implementations '
                'of the classification pipeline and change detection function. '
                'All AI-generated code was reviewed, tested against unit tests '
                'and synthetic datasets with known ground truth, and modified '
                'where necessary before incorporation into the pipeline.'
            ),
            'prompt_framework': 'CAPE (Context, Action, Parameters, Expected output)',
            'human_review': True,
            'tests_passed': True
        },
        'validation': {
            'unit_tests': 'All tests passed (pytest, 12 test functions)',
            'accuracy_assessment': {
                'method': 'Confusion matrix against synthetic ground truth',
                'overall_accuracy': round(float(oa), 4),
                'cohens_kappa': round(float(kappa), 4)
            },
            'uncertainty': {
                'method': 'Monte Carlo propagation (n=300 simulations)',
                'ndvi_mean_sigma': round(float(np.nanmean(ndvi_uncertainty)), 4)
            }
        }
    }

    if output_path:
        with open(output_path, 'w') as f:
            json.dump(report, f, indent=2)
        print(f'Report saved to: {output_path}')

    return report


report = generate_pipeline_report(DATA_DIR / 'pipeline_reproducibility_report.json')
print(json.dumps(report, indent=2))

---

## 8. Integration Test: End-to-End Pipeline Smoke Test

The final step is a lightweight **integration test** that runs the complete pipeline on the synthetic dataset and verifies that the outputs meet minimum quality thresholds.

In [ ]:
%%ipytest -v

def test_integration_full_pipeline_on_synthetic_data():
    """
    Integration smoke test: run the full processing chain on synthetic data
    and verify that outputs meet minimum quality thresholds.
    """
    bands_test, gt = make_landcover_reflectance(n_rows=50, n_cols=50, seed=99)

    # 1. Spectral index computation
    ndvi = safe_index(bands_test['nir'], bands_test['red'])
    ndwi = safe_index(bands_test['green'], bands_test['nir'])

    assert ndvi.shape == (50, 50), 'NDVI shape mismatch'
    assert np.nanmax(ndvi) <= 1.0 + 1e-6, 'NDVI exceeds 1.0'
    assert np.nanmin(ndvi) >= -1.0 - 1e-6, 'NDVI below -1.0'

    # 2. Vegetation pixels (top-right quadrant in synthetic scene)
    veg_region = ndvi[:25, 25:]
    assert np.nanmean(veg_region) > 0.4, 'Vegetation NDVI should be > 0.4'

    # 3. Water pixels (top-left quadrant)
    water_region = ndvi[:25, :25]
    assert np.nanmean(water_region) < 0.1, 'Water NDVI should be < 0.1'

    # 4. Change detection returns correct shape
    ndvi_t1 = ndvi.copy()
    ndvi_t2 = ndvi + 0.2  # simulate greening
    ndvi_t2 = np.clip(ndvi_t2, -1, 1)
    delta, change = detect_ndvi_change(ndvi_t1, ndvi_t2, threshold=0.15)
    assert delta.shape == (50, 50)
    assert change.shape == (50, 50)

    # 5. Greening class should dominate (delta = 0.2 > threshold 0.15)
    valid = change != -9
    pct_green = (change[valid] == 1).mean()
    assert pct_green > 0.5, f'Expected >50% greening, got {pct_green:.2f}'

    print('✅ Integration test passed!')


def test_integration_reflectance_scaling():
    """
    Verify that dividing raw Landsat integers by 10000 gives sensible
    reflectance values (0–1 range after clipping).
    """
    raw_ints = np.array([[5000, 8000, 500, 0, 10001, -1]])
    scaled = raw_ints.astype(float) / 10000.0
    clipped = clip_reflectance(scaled)

    assert clipped.min() == 0.0, 'Min should be 0 after clipping'
    assert clipped.max() == 1.0, 'Max should be 1 after clipping'
    np.testing.assert_allclose(clipped[0, 0], 0.5)  # 5000/10000
    np.testing.assert_allclose(clipped[0, 1], 0.8)  # 8000/10000
    np.testing.assert_allclose(clipped[0, 2], 0.05)  # 500/10000

    print('✅ Reflectance scaling test passed!')

---

## ✅ Part 3 Summary — Full Course Wrap-Up

### What you have accomplished in this 3-part summer school:

**Part 1 — Data Acquisition:**
- ✅ Connected to Planetary Computer STAC API
- ✅ Downloaded Landsat Collection 2 and Sentinel-2 L2A imagery
- ✅ Computed NDVI, NDWI, NDBI spectral indices
- ✅ Compared Landsat (30 m) vs Sentinel-2 (10 m) outputs

**Part 2 — AI Agents:**
- ✅ Understood the AI Evangelist mindset
- ✅ Applied the CAPE framework for scientific prompts
- ✅ Built a K-Means land cover classification pipeline with AI assistance
- ✅ Implemented NDVI change detection for multi-temporal analysis
- ✅ Developed validation helper functions for AI code review

**Part 3 — Testing and Validation:**
- ✅ Wrote `pytest` unit tests for all core functions
- ✅ Built synthetic test fixtures with known ground truth
- ✅ Computed a full accuracy assessment (OA, Kappa, confusion matrix)
- ✅ Quantified NDVI uncertainty via Monte Carlo simulation
- ✅ Generated a reproducibility report for archiving
- ✅ Ran end-to-end integration tests

---

### 🎓 The AI Evangelist Manifesto

> *"I use AI coding agents as a force multiplier for rigorous science. AI writes the first draft of code; I write the tests. AI generates options; I evaluate them with domain expertise. AI accelerates; I steer. The result is better science, faster."*

---

### 📚 Further Reading

- **STAC specification**: https://stacspec.org/
- **Microsoft Planetary Computer**: https://planetarycomputer.microsoft.com/
- **Landsat Collection 2 guide**: https://www.usgs.gov/landsat-missions/landsat-collection-2
- **Sentinel-2 User Handbook**: https://sentinel.esa.int/documents/247904/685211/Sentinel-2_User_Handbook
- **Good Scientific Practice with AI tools**: https://doi.org/10.1038/s41586-023-06221-2

---

**Thank you for attending IGARSS 2026! Questions? Contact: eoughton@gmu.edu**